# U.S. Airline Operational Performance Analytics 2025
## 01 — Data Preprocessing


## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import zipfile
import re
from pathlib import Path

print(f"Pandas version: {pd.__version__}")
print("Libraries imported successfully.")


Pandas version: 3.0.3
Libraries imported successfully.


## 2. Detect Project Folder

In [ ]:
current_path = Path.cwd().resolve()

if (current_path / "data" / "raw").exists():
    project_root = current_path
elif (current_path.parent / "data" / "raw").exists():
    project_root = current_path.parent
else:
    raise FileNotFoundError(
        "Folder data/raw tidak ditemukan. Pastikan struktur folder project sudah dibuat sesuai petunjuk."
    )

raw_path = project_root / "data" / "raw"
processed_path = project_root / "data" / "processed"

processed_path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")
print(f"Raw data path: {raw_path}")
print(f"Processed data path: {processed_path}")


## 3. Find and Validate 12 ZIP Files

In [3]:
zip_files = list(raw_path.glob("*.zip"))

print(f"Total ZIP files found: {len(zip_files)}")

if len(zip_files) != 12:
    raise ValueError(
        f"Expected 12 ZIP files for January-December 2025, but found {len(zip_files)}."
    )


Total ZIP files found: 12


In [4]:
def get_month_from_filename(file_path):
    match = re.search(r"_2025_(\d{1,2})\.zip$", file_path.name)

    if match is None:
        raise ValueError(f"Month number not found in filename: {file_path.name}")

    return int(match.group(1))

zip_files = sorted(zip_files, key=get_month_from_filename)

months_found = [get_month_from_filename(file) for file in zip_files]

print(f"Months found: {months_found}")

if months_found != list(range(1, 13)):
    raise ValueError(
        f"Expected months 1-12, but found: {months_found}"
    )

for file in zip_files:
    print(f"Month {get_month_from_filename(file):02d}: {file.name}")


Months found: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Month 01: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_1.zip
Month 02: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_2.zip
Month 03: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_3.zip
Month 04: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_4.zip
Month 05: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_5.zip
Month 06: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_6.zip
Month 07: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_7.zip
Month 08: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_8.zip
Month 09: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_9.zip
Month 10: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_10.zip
Month 11: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_11.zip
Month 12: On_Time_Reporting_Carrier_On_Time_Performance_1987_pre

## 4. Inspect First ZIP and Raw Schema

In [5]:
first_zip = zip_files[0]

with zipfile.ZipFile(first_zip, "r") as zip_ref:
    file_names = zip_ref.namelist()

print(f"ZIP file: {first_zip.name}")
print(f"Files inside ZIP:")

for file_name in file_names:
    print(f"- {file_name}")


ZIP file: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_1.zip
Files inside ZIP:
- On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_1.csv
- readme.html


In [6]:
with zipfile.ZipFile(first_zip, "r") as zip_ref:
    csv_files = [
        file_name
        for file_name in zip_ref.namelist()
        if file_name.lower().endswith(".csv")
    ]

if len(csv_files) != 1:
    raise ValueError(
        f"Expected exactly 1 CSV inside {first_zip.name}, but found {len(csv_files)}."
    )

with zipfile.ZipFile(first_zip, "r") as zip_ref:
    with zip_ref.open(csv_files[0]) as csv_file:
        df_sample = pd.read_csv(csv_file, nrows=5)

print(f"Sample rows: {df_sample.shape[0]}")
print(f"Total raw columns: {df_sample.shape[1]}")


Sample rows: 5
Total raw columns: 110


In [7]:
print("Raw column names:")

for column in df_sample.columns:
    print(column)


Raw column names:
Year
Quarter
Month
DayofMonth
DayOfWeek
FlightDate
Reporting_Airline
DOT_ID_Reporting_Airline
IATA_CODE_Reporting_Airline
Tail_Number
Flight_Number_Reporting_Airline
OriginAirportID
OriginAirportSeqID
OriginCityMarketID
Origin
OriginCityName
OriginState
OriginStateFips
OriginStateName
OriginWac
DestAirportID
DestAirportSeqID
DestCityMarketID
Dest
DestCityName
DestState
DestStateFips
DestStateName
DestWac
CRSDepTime
DepTime
DepDelay
DepDelayMinutes
DepDel15
DepartureDelayGroups
DepTimeBlk
TaxiOut
WheelsOff
WheelsOn
TaxiIn
CRSArrTime
ArrTime
ArrDelay
ArrDelayMinutes
ArrDel15
ArrivalDelayGroups
ArrTimeBlk
Cancelled
CancellationCode
Diverted
CRSElapsedTime
ActualElapsedTime
AirTime
Flights
Distance
DistanceGroup
CarrierDelay
WeatherDelay
NASDelay
SecurityDelay
LateAircraftDelay
FirstDepTime
TotalAddGTime
LongestAddGTime
DivAirportLandings
DivReachedDest
DivActualElapsedTime
DivArrDelay
DivDistance
Div1Airport
Div1AirportID
Div1AirportSeqID
Div1WheelsOn
Div1TotalGTime
Div1

## 5. Select Relevant Columns

In [8]:
selected_columns = [
    # Flight identity
    "FlightDate",
    "Reporting_Airline",
    "DOT_ID_Reporting_Airline",
    "Tail_Number",
    "Flight_Number_Reporting_Airline",

    # Origin
    "OriginAirportID",
    "Origin",
    "OriginCityName",
    "OriginState",
    "OriginStateName",

    # Destination
    "DestAirportID",
    "Dest",
    "DestCityName",
    "DestState",
    "DestStateName",

    # Departure operation
    "CRSDepTime",
    "DepTime",
    "DepDelay",
    "DepDelayMinutes",
    "DepDel15",
    "DepTimeBlk",
    "TaxiOut",
    "WheelsOff",

    # Arrival operation
    "WheelsOn",
    "TaxiIn",
    "CRSArrTime",
    "ArrTime",
    "ArrDelay",
    "ArrDelayMinutes",
    "ArrDel15",
    "ArrTimeBlk",

    # Disruption
    "Cancelled",
    "CancellationCode",
    "Diverted",

    # Duration and distance
    "CRSElapsedTime",
    "ActualElapsedTime",
    "AirTime",
    "Distance",

    # Delay causes
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay",

    # Diversion summary
    "DivAirportLandings"
]

print(f"Selected columns: {len(selected_columns)}")


Selected columns: 44


In [9]:
missing_selected_columns = [
    column
    for column in selected_columns
    if column not in df_sample.columns
]

if len(missing_selected_columns) > 0:
    raise ValueError(
        f"Selected columns not found in raw data: {missing_selected_columns}"
    )

print("All selected columns are available in the raw dataset.")


All selected columns are available in the raw dataset.


## 6. Define Data Types and Processing Settings

In [10]:
chunk_size = 100_000

numeric_columns = [
    "DOT_ID_Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "OriginAirportID",
    "DestAirportID",
    "CRSDepTime",
    "DepTime",
    "DepDelay",
    "DepDelayMinutes",
    "DepDel15",
    "TaxiOut",
    "WheelsOff",
    "WheelsOn",
    "TaxiIn",
    "CRSArrTime",
    "ArrTime",
    "ArrDelay",
    "ArrDelayMinutes",
    "ArrDel15",
    "Cancelled",
    "Diverted",
    "CRSElapsedTime",
    "ActualElapsedTime",
    "AirTime",
    "Distance",
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay",
    "DivAirportLandings"
]

text_columns = [
    "Reporting_Airline",
    "Tail_Number",
    "Origin",
    "OriginCityName",
    "OriginState",
    "OriginStateName",
    "Dest",
    "DestCityName",
    "DestState",
    "DestStateName",
    "DepTimeBlk",
    "ArrTimeBlk",
    "CancellationCode"
]

candidate_key_columns = [
    "FlightDate",
    "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime"
]

delay_cause_columns = [
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay"
]

print(f"Chunk size: {chunk_size:,} rows")
print(f"Candidate key columns: {candidate_key_columns}")


Chunk size: 100,000 rows
Candidate key columns: ['FlightDate', 'Reporting_Airline', 'Flight_Number_Reporting_Airline', 'Origin', 'Dest', 'CRSDepTime']


## 7. Process January–December 2025

Proses dilakukan **per bulan dan per chunk** supaya lebih aman untuk RAM laptop.

Output akan dibuat sebagai:

```text
data/processed/
├── airline_2025_01_processed.csv.gz
├── airline_2025_02_processed.csv.gz
├── ...
└── airline_2025_12_processed.csv.gz
```

File `.csv.gz` tetap CSV, tetapi dikompres supaya ukuran storage lebih kecil.


In [11]:
audit_results = []

for zip_path in zip_files:
    month = get_month_from_filename(zip_path)
    output_file = processed_path / f"airline_2025_{month:02d}_processed.csv.gz"

    print("")
    print("=" * 70)
    print(f"Processing month: {month:02d}")
    print(f"Source file: {zip_path.name}")
    print(f"Output file: {output_file.name}")

    total_rows = 0
    total_duplicates = 0
    total_cancelled = 0
    total_diverted = 0
    total_valid_arrivals = 0
    total_arrival_delayed_15 = 0
    total_severe_delays = 0

    seen_hashes = set()
    first_chunk = True

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        csv_files = [
            file_name
            for file_name in zip_ref.namelist()
            if file_name.lower().endswith(".csv")
        ]

        if len(csv_files) != 1:
            raise ValueError(
                f"Expected exactly 1 CSV inside {zip_path.name}, but found {len(csv_files)}."
            )

        with zip_ref.open(csv_files[0]) as csv_file:
            reader = pd.read_csv(
                csv_file,
                usecols=selected_columns,
                chunksize=chunk_size,
                low_memory=False
            )

            for chunk_number, df in enumerate(reader, start=1):
                rows_before = len(df)
                total_rows += rows_before

                # -----------------------------
                # Basic cleaning
                # -----------------------------
                df["FlightDate"] = pd.to_datetime(
                    df["FlightDate"],
                    errors="coerce"
                )

                for column in numeric_columns:
                    df[column] = pd.to_numeric(
                        df[column],
                        errors="coerce"
                    )

                for column in text_columns:
                    df[column] = df[column].astype("string").str.strip()

                # -----------------------------
                # Duplicate check
                # Candidate key:
                # date + airline + flight no + origin + destination + scheduled departure
                # -----------------------------
                current_hashes = pd.util.hash_pandas_object(
                    df[candidate_key_columns],
                    index=False
                ).astype("uint64")

                duplicate_mask = current_hashes.isin(seen_hashes)
                duplicate_count = int(duplicate_mask.sum())

                if duplicate_count > 0:
                    df = df.loc[~duplicate_mask].copy()
                    current_hashes = current_hashes.loc[~duplicate_mask]

                seen_hashes.update(current_hashes.tolist())
                total_duplicates += duplicate_count

                # -----------------------------
                # Date features
                # -----------------------------
                df["Year"] = df["FlightDate"].dt.year.astype("Int16")
                df["Quarter"] = df["FlightDate"].dt.quarter.astype("Int8")
                df["MonthNumber"] = df["FlightDate"].dt.month.astype("Int8")
                df["MonthName"] = df["FlightDate"].dt.month_name()
                df["DayOfMonth"] = df["FlightDate"].dt.day.astype("Int8")
                df["DayOfWeekNumber"] = df["FlightDate"].dt.dayofweek.add(1).astype("Int8")
                df["DayName"] = df["FlightDate"].dt.day_name()
                df["WeekendFlag"] = (
                    df["FlightDate"].dt.dayofweek >= 5
                ).astype("Int8")

                # -----------------------------
                # Route feature
                # -----------------------------
                df["Route"] = (
                    df["Origin"].fillna("Unknown")
                    + " - "
                    + df["Dest"].fillna("Unknown")
                )

                # -----------------------------
                # Scheduled departure hour
                # BTS time uses HHMM format
                # Example: 930 = 09:30
                # -----------------------------
                scheduled_dep_hour = (
                    df["CRSDepTime"] // 100
                ).astype("Int64")

                scheduled_dep_hour = scheduled_dep_hour.replace(24, 0)

                df["ScheduledDepHour"] = scheduled_dep_hour

                df["DeparturePeriod"] = pd.cut(
                    df["ScheduledDepHour"],
                    bins=[-1, 5, 11, 17, 23],
                    labels=[
                        "Night",
                        "Morning",
                        "Afternoon",
                        "Evening"
                    ]
                )

                # -----------------------------
                # Operational flags
                # -----------------------------
                valid_arrival_mask = (
                    (df["Cancelled"] == 0)
                    & (df["Diverted"] == 0)
                    & (df["ArrDelay"].notna())
                )

                df["ValidArrivalFlag"] = 0
                df.loc[valid_arrival_mask, "ValidArrivalFlag"] = 1

                df["ArrivalDelayed15Flag"] = pd.Series(
                    pd.NA,
                    index=df.index,
                    dtype="Int8"
                )

                df.loc[
                    valid_arrival_mask,
                    "ArrivalDelayed15Flag"
                ] = (
                    df.loc[valid_arrival_mask, "ArrDelay"] >= 15
                ).astype("Int8")

                df["OnTimeArrivalFlag"] = pd.Series(
                    pd.NA,
                    index=df.index,
                    dtype="Int8"
                )

                df.loc[
                    valid_arrival_mask,
                    "OnTimeArrivalFlag"
                ] = (
                    df.loc[valid_arrival_mask, "ArrDelay"] < 15
                ).astype("Int8")

                df["SevereDelayFlag"] = pd.Series(
                    pd.NA,
                    index=df.index,
                    dtype="Int8"
                )

                df.loc[
                    valid_arrival_mask,
                    "SevereDelayFlag"
                ] = (
                    df.loc[valid_arrival_mask, "ArrDelay"] >= 60
                ).astype("Int8")

                # -----------------------------
                # Arrival status
                # -----------------------------
                df["ArrivalStatus"] = "Unknown"

                df.loc[
                    df["Cancelled"] == 1,
                    "ArrivalStatus"
                ] = "Cancelled"

                df.loc[
                    (df["Cancelled"] == 0)
                    & (df["Diverted"] == 1),
                    "ArrivalStatus"
                ] = "Diverted"

                df.loc[
                    valid_arrival_mask
                    & (df["ArrDelay"] < 15),
                    "ArrivalStatus"
                ] = "On Time"

                df.loc[
                    valid_arrival_mask
                    & (df["ArrDelay"] >= 15),
                    "ArrivalStatus"
                ] = "Delayed 15+ min"

                # -----------------------------
                # Delay severity
                # -----------------------------
                df["DelaySeverity"] = "Not Applicable"

                df.loc[
                    valid_arrival_mask & (df["ArrDelay"] < 15),
                    "DelaySeverity"
                ] = "On Time (<15 min)"

                df.loc[
                    valid_arrival_mask
                    & (df["ArrDelay"] >= 15)
                    & (df["ArrDelay"] < 30),
                    "DelaySeverity"
                ] = "15-29 min"

                df.loc[
                    valid_arrival_mask
                    & (df["ArrDelay"] >= 30)
                    & (df["ArrDelay"] < 60),
                    "DelaySeverity"
                ] = "30-59 min"

                df.loc[
                    valid_arrival_mask
                    & (df["ArrDelay"] >= 60)
                    & (df["ArrDelay"] < 120),
                    "DelaySeverity"
                ] = "60-119 min"

                df.loc[
                    valid_arrival_mask
                    & (df["ArrDelay"] >= 120),
                    "DelaySeverity"
                ] = "120+ min"

                # -----------------------------
                # Delay cause total
                # Original cause columns keep their NaN values.
                # Only the derived total treats missing cause as 0.
                # -----------------------------
                df["TotalCauseDelayMinutes"] = (
                    df[delay_cause_columns]
                    .fillna(0)
                    .sum(axis=1)
                )

                # -----------------------------
                # Net delay change
                # Positive value = arrival delay lower than departure delay
                # Negative value = delay increased by arrival
                # -----------------------------
                df["NetDelayChangeMinutes"] = (
                    df["DepDelay"] - df["ArrDelay"]
                )

                # -----------------------------
                # Distance band
                # -----------------------------
                df["DistanceBand"] = pd.cut(
                    df["Distance"],
                    bins=[
                        -np.inf,
                        500,
                        1000,
                        1500,
                        2500,
                        np.inf
                    ],
                    labels=[
                        "0-500 miles",
                        "501-1000 miles",
                        "1001-1500 miles",
                        "1501-2500 miles",
                        "2500+ miles"
                    ]
                )

                # -----------------------------
                # Monthly audit counters
                # -----------------------------
                total_cancelled += int(
                    (df["Cancelled"] == 1).sum()
                )

                total_diverted += int(
                    (df["Diverted"] == 1).sum()
                )

                total_valid_arrivals += int(
                    df["ValidArrivalFlag"].sum()
                )

                total_arrival_delayed_15 += int(
                    df["ArrivalDelayed15Flag"].fillna(0).sum()
                )

                total_severe_delays += int(
                    df["SevereDelayFlag"].fillna(0).sum()
                )

                # -----------------------------
                # Save processed chunk
                # -----------------------------
                write_mode = "w" if first_chunk else "a"

                df.to_csv(
                    output_file,
                    index=False,
                    mode=write_mode,
                    header=first_chunk,
                    compression="gzip",
                    date_format="%Y-%m-%d"
                )

                first_chunk = False

                print(
                    f"Month {month:02d} | "
                    f"Chunk {chunk_number} | "
                    f"Rows read: {rows_before:,} | "
                    f"Duplicates removed: {duplicate_count:,}"
                )

    arrival_delay_rate = (
        total_arrival_delayed_15 / total_valid_arrivals * 100
        if total_valid_arrivals > 0
        else np.nan
    )

    cancellation_rate = (
        total_cancelled / (total_rows - total_duplicates) * 100
        if (total_rows - total_duplicates) > 0
        else np.nan
    )

    diversion_rate = (
        total_diverted / (total_rows - total_duplicates) * 100
        if (total_rows - total_duplicates) > 0
        else np.nan
    )

    severe_delay_rate = (
        total_severe_delays / total_valid_arrivals * 100
        if total_valid_arrivals > 0
        else np.nan
    )

    audit_results.append({
        "Month": month,
        "RowsRead": total_rows,
        "DuplicatesRemoved": total_duplicates,
        "RowsSaved": total_rows - total_duplicates,
        "CancelledFlights": total_cancelled,
        "DivertedFlights": total_diverted,
        "ValidArrivals": total_valid_arrivals,
        "ArrivalDelayed15Flights": total_arrival_delayed_15,
        "SevereDelayFlights": total_severe_delays,
        "ArrivalDelayRatePct": arrival_delay_rate,
        "CancellationRatePct": cancellation_rate,
        "DiversionRatePct": diversion_rate,
        "SevereDelayRatePct": severe_delay_rate
    })

    print(f"Finished month: {month:02d}")
    print(f"Rows read: {total_rows:,}")
    print(f"Duplicates removed: {total_duplicates:,}")
    print(f"Rows saved: {total_rows - total_duplicates:,}")
    print(f"Arrival delay rate: {arrival_delay_rate:.2f}%")
    print(f"Cancellation rate: {cancellation_rate:.2f}%")
    print(f"Diversion rate: {diversion_rate:.2f}%")



Processing month: 01
Source file: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_1.zip
Output file: airline_2025_01_processed.csv.gz
Month 01 | Chunk 1 | Rows read: 100,000 | Duplicates removed: 0
Month 01 | Chunk 2 | Rows read: 100,000 | Duplicates removed: 0
Month 01 | Chunk 3 | Rows read: 100,000 | Duplicates removed: 0
Month 01 | Chunk 4 | Rows read: 100,000 | Duplicates removed: 0
Month 01 | Chunk 5 | Rows read: 100,000 | Duplicates removed: 0
Month 01 | Chunk 6 | Rows read: 39,747 | Duplicates removed: 0
Finished month: 01
Rows read: 539,747
Duplicates removed: 0
Rows saved: 539,747
Arrival delay rate: 18.79%
Cancellation rate: 3.02%
Diversion rate: 0.22%

Processing month: 02
Source file: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_2.zip
Output file: airline_2025_02_processed.csv.gz
Month 02 | Chunk 1 | Rows read: 100,000 | Duplicates removed: 0
Month 02 | Chunk 2 | Rows read: 100,000 | Duplicates removed: 0
Month 02 | Chunk 3 | Rows read: 1

## 8. Create Monthly Audit Summary

Bagian ini merangkum hasil preprocessing per bulan agar kita bisa mengecek apakah output masuk akal sebelum lanjut ke EDA.


In [12]:
audit_df = pd.DataFrame(audit_results)

audit_df = audit_df.sort_values("Month").reset_index(drop=True)

audit_df


,Month,RowsRead,DuplicatesRemoved,RowsSaved,CancelledFlights,DivertedFlights,ValidArrivals,ArrivalDelayed15Flights,SevereDelayFlights,ArrivalDelayRatePct,CancellationRatePct,DiversionRatePct,SevereDelayRatePct
0,1,539747,0,539747,16312,1166,522269,98130,33245,18.789168,3.022157,0.216027,6.365494
1,2,504884,0,504884,7405,1003,496476,103102,36874,20.766764,1.466674,0.198659,7.427147
2,3,600872,0,600872,6923,1648,592301,116034,41895,19.590377,1.152159,0.274268,7.073262
3,4,583950,0,583950,4914,1306,577730,113604,39001,19.663857,0.841510,0.223649,6.750731
4,5,605648,0,605648,6344,1730,597574,141038,52142,23.601763,1.047473,0.285644,8.725614
5,6,611575,0,611575,9736,2367,599472,169405,67845,28.259035,1.591955,0.387033,11.317459
6,7,631428,0,631428,15473,3144,612811,177012,74051,28.885252,2.450477,0.497919,12.083824
7,8,602378,0,602378,6483,2162,593733,133920,48818,22.555593,1.076235,0.358911,8.222214
8,9,562439,0,562439,2886,1224,558328,92859,30271,16.631622,0.513122,0.217624,5.421723
9,10,605844,0,605844,3204,1070,601570,122251,38068,20.321991,0.528849,0.176613,6.328108


In [ ]:
audit_summary_file = processed_path / "preprocessing_audit_2025.csv"

audit_df.to_csv(
    audit_summary_file,
    index=False
)

print(f"Audit summary saved to: {audit_summary_file}")


## 9. Full-Year Validation

In [14]:
total_rows_2025 = int(audit_df["RowsSaved"].sum())
total_cancelled_2025 = int(audit_df["CancelledFlights"].sum())
total_diverted_2025 = int(audit_df["DivertedFlights"].sum())
total_valid_arrivals_2025 = int(audit_df["ValidArrivals"].sum())
total_delayed_15_2025 = int(audit_df["ArrivalDelayed15Flights"].sum())
total_severe_delays_2025 = int(audit_df["SevereDelayFlights"].sum())
total_duplicates_2025 = int(audit_df["DuplicatesRemoved"].sum())

arrival_delay_rate_2025 = (
    total_delayed_15_2025 / total_valid_arrivals_2025 * 100
)

on_time_rate_2025 = (
    100 - arrival_delay_rate_2025
)

cancellation_rate_2025 = (
    total_cancelled_2025 / total_rows_2025 * 100
)

diversion_rate_2025 = (
    total_diverted_2025 / total_rows_2025 * 100
)

severe_delay_rate_2025 = (
    total_severe_delays_2025 / total_valid_arrivals_2025 * 100
)

print(f"Total processed rows: {total_rows_2025:,}")
print(f"Total duplicates removed: {total_duplicates_2025:,}")
print(f"Valid arrivals: {total_valid_arrivals_2025:,}")
print(f"Arrival delayed 15+ min: {total_delayed_15_2025:,}")
print(f"On-time arrival rate: {on_time_rate_2025:.2f}%")
print(f"Arrival delay rate: {arrival_delay_rate_2025:.2f}%")
print(f"Cancellation rate: {cancellation_rate_2025:.2f}%")
print(f"Diversion rate: {diversion_rate_2025:.2f}%")
print(f"Severe delay rate: {severe_delay_rate_2025:.2f}%")


Total processed rows: 7,001,619
Total duplicates removed: 0
Valid arrivals: 6,879,484
Arrival delayed 15+ min: 1,534,638
On-time arrival rate: 77.69%
Arrival delay rate: 22.31%
Cancellation rate: 1.47%
Diversion rate: 0.28%
Severe delay rate: 8.18%


## 10. Check Processed Files

Pastikan ada **12 processed files** dan **1 audit summary**.


In [15]:
processed_files = sorted(
    processed_path.glob("airline_2025_*_processed.csv.gz")
)

print(f"Total processed monthly files: {len(processed_files)}")

for file in processed_files:
    file_size_mb = file.stat().st_size / (1024 ** 2)
    print(f"{file.name} | {file_size_mb:.2f} MB")

if len(processed_files) != 12:
    raise ValueError(
        f"Expected 12 processed files, but found {len(processed_files)}."
    )


Total processed monthly files: 12
airline_2025_01_processed.csv.gz | 25.69 MB
airline_2025_02_processed.csv.gz | 24.03 MB
airline_2025_03_processed.csv.gz | 28.88 MB
airline_2025_04_processed.csv.gz | 27.78 MB
airline_2025_05_processed.csv.gz | 29.11 MB
airline_2025_06_processed.csv.gz | 29.38 MB
airline_2025_07_processed.csv.gz | 30.32 MB
airline_2025_08_processed.csv.gz | 29.13 MB
airline_2025_09_processed.csv.gz | 26.65 MB
airline_2025_10_processed.csv.gz | 29.10 MB
airline_2025_11_processed.csv.gz | 27.31 MB
airline_2025_12_processed.csv.gz | 28.74 MB


## 11. Read Small Sample from Processed Data

In [16]:
sample_processed_file = processed_files[0]

df_processed_sample = pd.read_csv(
    sample_processed_file,
    nrows=10,
    compression="gzip"
)

print(f"Sample processed rows: {df_processed_sample.shape[0]}")
print(f"Processed columns: {df_processed_sample.shape[1]}")

df_processed_sample.head()


Sample processed rows: 10
Processed columns: 64


,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,OriginAirportID,Origin,OriginCityName,OriginState,OriginStateName,...,DeparturePeriod,ValidArrivalFlag,ArrivalDelayed15Flag,OnTimeArrivalFlag,SevereDelayFlag,ArrivalStatus,DelaySeverity,TotalCauseDelayMinutes,NetDelayChangeMinutes,DistanceBand
0,2025-01-01,AA,19805,N104NN,1,12478,JFK,"New York, NY",NY,New York,...,Morning,1,0,1,0,On Time,On Time (<15 min),0.0,4.0,1501-2500 miles
1,2025-01-02,AA,19805,N110AN,1,12478,JFK,"New York, NY",NY,New York,...,Morning,1,0,1,0,On Time,On Time (<15 min),0.0,-9.0,1501-2500 miles
2,2025-01-03,AA,19805,N106NN,1,12478,JFK,"New York, NY",NY,New York,...,Morning,1,0,1,0,On Time,On Time (<15 min),0.0,10.0,1501-2500 miles
3,2025-01-04,AA,19805,N117AN,1,12478,JFK,"New York, NY",NY,New York,...,Morning,1,0,1,0,On Time,On Time (<15 min),0.0,3.0,1501-2500 miles
4,2025-01-05,AA,19805,N104NN,1,12478,JFK,"New York, NY",NY,New York,...,Morning,1,0,1,0,On Time,On Time (<15 min),0.0,3.0,1501-2500 miles


## 12. Check Final Processed Columns

In [17]:
for column in df_processed_sample.columns:
    print(column)


FlightDate
Reporting_Airline
DOT_ID_Reporting_Airline
Tail_Number
Flight_Number_Reporting_Airline
OriginAirportID
Origin
OriginCityName
OriginState
OriginStateName
DestAirportID
Dest
DestCityName
DestState
DestStateName
CRSDepTime
DepTime
DepDelay
DepDelayMinutes
DepDel15
DepTimeBlk
TaxiOut
WheelsOff
WheelsOn
TaxiIn
CRSArrTime
ArrTime
ArrDelay
ArrDelayMinutes
ArrDel15
ArrTimeBlk
Cancelled
CancellationCode
Diverted
CRSElapsedTime
ActualElapsedTime
AirTime
Distance
CarrierDelay
WeatherDelay
NASDelay
SecurityDelay
LateAircraftDelay
DivAirportLandings
Year
Quarter
MonthNumber
MonthName
DayOfMonth
DayOfWeekNumber
DayName
WeekendFlag
Route
ScheduledDepHour
DeparturePeriod
ValidArrivalFlag
ArrivalDelayed15Flag
OnTimeArrivalFlag
SevereDelayFlag
ArrivalStatus
DelaySeverity
TotalCauseDelayMinutes
NetDelayChangeMinutes
DistanceBand
